In [ ]:
import csv
import importlib
import warnings
from collections import defaultdict

import matplotlib.pyplot as plt
import torch
from tensordict.nn import TensorDictModule
from torch import multiprocessing, nn
from torchrl.collectors import Collector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (
    Compose,
    DoubleToFloat,
    ObservationNorm,
    ParallelEnv,
    StepCounter,
    TransformedEnv,
)
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import MaskedCategorical, ProbabilisticActor, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm

import env as env_module

warnings.filterwarnings("ignore")
importlib.reload(env_module)

In [ ]:
is_fork = multiprocessing.get_start_method() == "fork"
device = (
    torch.device(0)
    if torch.cuda.is_available() and not is_fork
    else torch.device("cpu")
)

In [ ]:
## Hyperparameters

num_cells = 256
lr = 3e-4
max_grad_norm = 1.0

NUM_AGENTS = 2
frames_per_batch = 600
total_frames = 3_000_000

## PPO parameters

sub_batch_size = 64
num_epochs = 10
clip_epsilon = 0.2
gamma = 0.99
lmbda = 0.95
entropy_eps = 0.1

SAVE_EVERY = 50  # save checkpoint every N batches

In [ ]:
base_env = ParallelEnv(NUM_AGENTS, lambda: env_module.ZappyEnv())

In [ ]:
obs_size = env_module.OBSERVATION_SIZE

env = TransformedEnv(
    base_env,
    Compose(
        # obs already in [0,1] — identity normalization, no init_stats needed
        ObservationNorm(
            in_keys=["observation"],
            loc=torch.zeros(obs_size),
            scale=torch.ones(obs_size),
        ),
        DoubleToFloat(),
        StepCounter(),
    ),
)

In [ ]:
# loc=0, scale=1 : pas besoin d'init_stats, les obs sont déjà normalisées [0,1]
print("obs_size:", obs_size)
print("loc shape:", env.transform[0].loc.shape)
print("scale shape:", env.transform[0].scale.shape)

In [ ]:
print("normalization constant shape:", env.transform[0].loc.shape)

In [ ]:
print("observation_spec:", env.observation_spec)
print("reward_spec:", env.reward_spec)
print("input_spec:", env.input_spec)
print("action_spec (as defined by input_spec):", env.action_spec)

In [ ]:
print("num_envs:", base_env.num_workers)

In [ ]:
print("num_envs:", base_env.num_workers)

In [ ]:
rollout = env.rollout(3)
print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)

In [ ]:
actor_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(23, device=device),
)

In [ ]:
policy_module = TensorDictModule(
    actor_net, in_keys=["observation"], out_keys=["logits"]
)

In [ ]:
policy_module = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec,
    in_keys={"logits": "logits", "mask": "action_mask"},
    distribution_class=MaskedCategorical,
    return_log_prob=True,
)

In [ ]:
value_net = nn.Sequential(
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(num_cells, device=device),
    nn.Tanh(),
    nn.LazyLinear(1, device=device),
)

value_module = ValueOperator(
    module=value_net,
    in_keys=["observation"],
)

In [ ]:
td = env.reset()
print("Running policy:", policy_module(td))
print("Running value:", value_module(td))

In [ ]:
collector = Collector(
    env,
    policy_module,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    split_trajs=False,
    device=device,
)

In [ ]:
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

In [ ]:
advantage_module = GAE(
    gamma=gamma,
    lmbda=lmbda,
    value_network=value_module,
    average_gae=True,
    device=device,
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coeff=entropy_eps,
    # these keys match by default but we set this for completeness
    critic_coeff=1.0,
    loss_critic_type="smooth_l1",
)

optim = torch.optim.Adam(loss_module.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, total_frames // frames_per_batch, 1e-5
)

In [ ]:
import time

t0 = time.time()
td = env.reset()
print(f"reset en {time.time() - t0:.2f}s")

In [ ]:
def format_command_trace(commands, limit=12):
    if not commands:
        return "[]"
    if len(commands) <= limit:
        return "[" + " -> ".join(commands) + "]"
    head = " -> ".join(commands[: limit // 2])
    tail = " -> ".join(commands[-limit // 2 :])
    return f"[{head} -> ... -> {tail}]"

In [ ]:
logs = defaultdict(list)
pbar = tqdm(total=total_frames)
eval_str = ""
csv_path = "logs.csv"
last_eval_reward_mean = ""
last_eval_reward_sum = ""
last_eval_step_count = ""
last_eval_command_trace = ""
best_reward = float("-inf")

# We iterate over the collector until it reaches the total number of frames it was
# designed to collect:
with open(csv_path, "w", newline="") as csvfile:
    fieldnames = [
        "batch",
        "reward",
        "eval reward",
        "eval reward (sum)",
        "step_count",
        "lr",
        "commands",
        "eval commands",
    ]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for i, tensordict_data in enumerate(collector):
        batch_idx = i + 1
        batch_commands = [
            env_module.COMMANDS[int(action)]
            for action in tensordict_data["action"].reshape(-1).tolist()
        ]
        command_trace = format_command_trace(batch_commands)

        # we now have a batch of data to work with. Let's learn something from it.
        for _ in range(num_epochs):
            # We'll need an "advantage" signal to make PPO work.
            # We re-compute it at each epoch as its value depends on the value
            # network which is updated in the inner loop.
            advantage_module(tensordict_data)
            data_view = tensordict_data.reshape(-1)
            replay_buffer.extend(data_view.cpu())
            for _ in range(frames_per_batch // sub_batch_size):
                subdata = replay_buffer.sample(sub_batch_size)
                loss_vals = loss_module(subdata.to(device))
                loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
                )

                # Optimization: backward, grad clipping and optimization step
                loss_value.backward()
                # this is not strictly mandatory but it's good practice to keep
                # your gradient norm bounded
                torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_grad_norm)
                optim.step()
                optim.zero_grad()

        current_reward = tensordict_data["next", "reward"].mean().item()
        current_step_count = tensordict_data["step_count"].max().item()
        current_lr = optim.param_groups[0]["lr"]

        logs["reward"].append(current_reward)
        logs["step_count"].append(current_step_count)
        logs["lr"].append(current_lr)
        pbar.update(tensordict_data.numel())

        cum_reward_str = (
            f"average reward={current_reward: 4.4f} (init={logs['reward'][0]: 4.4f})"
        )
        stepcount_str = f"step count (max): {current_step_count}"
        lr_str = f"lr policy: {current_lr: 4.4f}"

        if i % 10 == 0:
            # We evaluate the policy once every 10 batches of data.
            # Evaluation is rather simple: execute the policy without exploration
            # (take the expected value of the action distribution) for a given
            # number of steps (1000, which is our ``env`` horizon).
            # The ``rollout`` method of the ``env`` can take a policy as argument:
            # it will then execute this policy at each step.
            with set_exploration_type(ExplorationType.RANDOM), torch.no_grad():
                # execute a rollout with the trained policy
                eval_rollout = env.rollout(1000, policy_module)
                current_eval_reward = eval_rollout["next", "reward"].mean().item()
                current_eval_reward_sum = eval_rollout["next", "reward"].sum().item()
                current_eval_step_count = eval_rollout["step_count"].max().item()
                eval_commands = [
                    env_module.COMMANDS[int(action)]
                    for action in eval_rollout["action"].reshape(-1).tolist()
                ]
                eval_command_trace = format_command_trace(eval_commands)
                logs["eval reward"].append(current_eval_reward)
                logs["eval reward (sum)"].append(current_eval_reward_sum)
                logs["eval step_count"].append(current_eval_step_count)
                eval_str = (
                    f"eval reward={current_eval_reward: 4.4f}, "
                    f"eval cumulative reward={current_eval_reward_sum: 4.4f}, "
                    f"eval step-count={current_eval_step_count}, "
                    f"commands={eval_command_trace}"
                )
                last_eval_reward_mean = current_eval_reward
                last_eval_reward_sum = current_eval_reward_sum
                last_eval_step_count = current_eval_step_count
                last_eval_command_trace = eval_command_trace
                del eval_rollout
            print(
                f"[eval] batch {batch_idx:04d} | "
                f"reward={current_reward: 4.4f} | "
                f"eval_reward={current_eval_reward: 4.4f} | "
                f"eval_sum={current_eval_reward_sum: 4.4f} | "
                f"eval_step_count={current_eval_step_count} | "
                f"commands={command_trace} | "
                f"eval_commands={eval_command_trace}",
                flush=True,
            )
        else:
            print(
                f"[train] batch {batch_idx:04d} | "
                f"reward={current_reward: 4.4f} | "
                f"step_count={current_step_count} | "
                f"lr={current_lr: 4.4f} | "
                f"commands={command_trace}",
                flush=True,
            )

        writer.writerow(
            {
                "batch": batch_idx,
                "reward": current_reward,
                "eval reward": last_eval_reward_mean,
                "eval reward (sum)": last_eval_reward_sum,
                "step_count": current_step_count,
                "lr": current_lr,
                "commands": command_trace,
                "eval commands": last_eval_command_trace,
            }
        )
        csvfile.flush()

        pbar.set_description(
            ", ".join([eval_str, cum_reward_str, stepcount_str, lr_str])
        )

        # We're also using a learning rate scheduler. Like the gradient clipping,
        # this is a nice-to-have but nothing necessary for PPO to work.
        scheduler.step()

        # Save checkpoint every SAVE_EVERY batches (overwrite → always keeps best current)
        if batch_idx % SAVE_EVERY == 0 or current_reward > best_reward:
            if current_reward > best_reward:
                best_reward = current_reward
            torch.save(policy_module.state_dict(), "ppo_policy.pth")
            torch.save(value_module.state_dict(), "ppo_value.pth")

In [ ]:
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(logs["reward"])
plt.title("training rewards (average)")
plt.subplot(2, 2, 2)
plt.plot(logs["step_count"])
plt.title("Max step count (training)")
plt.subplot(2, 2, 3)
plt.plot(logs["eval reward (sum)"])
plt.title("Return (test)")
plt.subplot(2, 2, 4)
plt.plot(logs["eval step_count"])
plt.title("Max step count (test)")
plt.show()

In [ ]:
# save agent
torch.save(policy_module.state_dict(), "ppo_policy.pth")
torch.save(value_module.state_dict(), "ppo_value.pth")